# Módulo 5: Gateway e Identidade -- Conectando APIs Externas

![Overview](../shared/img/05.drawio.png)

Neste módulo, você dará à Aria a habilidade de **gerenciar tarefas** conectando-a a uma REST API através do AgentCore Gateway, implementando segurança de identidade JWT.

---

### O que você vai aprender

| Tópico | Detalhes |
|---|---|
| **AgentCore Gateway** | Um endpoint MCP gerenciado que roteia chamadas de ferramentas para o backend |
| **Protocolo MCP** | Como o Gateway descobre e expõe a estrutura da sua API como ferramentas MCP |
| **Identidade e Auth JWT** | Como o modo `CUSTOM_JWT` valida tokens no AWS Cognito |
| **Fluxo de identidade** | O caminho percorrido pelo JWT, desde o Runtime até a API final |

## Atualização de Dependências

Esta etapa garante que os recursos na nuvem (Runtime, Memory) estejam em perfeita execução.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("05")

---

## A Arquitetura do Gateway

O AgentCore Gateway é um **endpoint gerenciado MCP** posicionado entre a IA e as APIs. Em vez de criar código de ferramenta (tool) para cada API, você aponta o Gateway para o serviço e ele:

1. **Auto-descobre** a estrutura REST (rotas, métodos, schemas)
2. **Gera ferramentas MCP dinâmicas** para a IA (`list_tasks`, `create_task`)
3. **Orquestra a autenticação** na ponte entre o agente e o alvo
4. **Roteia as requisições** disparando com credenciais AWS IAM puras

### Tipos de Targets Suportados

O Gateway aceita cinco tipos de provedores backend (targets):

| Target | Descrição do Ambiente AWS |
|---|---|
| **Lambda** | Invoca uma função AWS Lambda de forma direta |
| **API Gateway** | Descoberta automática de um API Gateway da AWS |
| **OpenAPI** | Ingestão manual de um manifesto OpenAPI/Swagger |
| **Smithy** | Modelagem Smithy corporativa |
| **Servidor MCP** | Conecta dinamicamente a outro servidor MCP (Model Context Protocol) |

### Modos de Autenticação

O Gateway suporta esses padrões operacionais de auth:

| Modo | Cenário AWS |
|---|---|
| `NONE` | APIs públicas, dev local e laboratórios liminares |
| `CUSTOM_JWT` | Chamadas em que a identidade de ponta a ponta do cliente carrega um JWT |
| `AWS_IAM` | Comunicação M2M assinalada pela AWS (Service-to-service) |

Aqui usaremos a malha estrita do `CUSTOM_JWT` gerada via Amazon Cognito.

> **Nota Operacional:** Deste laboratório em diante, o agente passará a exigir JWT em suas invocações. É esse JWT que blinda e isola a arquitetura no backend.

> **Documentation:** [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html)

---

## Entendendo a Propagação de Identidade

No modo `CUSTOM_JWT`, o barramento flui assim na rede:

1. O usuário faz o Login e o **Cognito emite um JWT**
2. O AgentCore Runtime **recebe o header Authorization** com o token
3. O código da Aria **repassa esse JWT** no cliente MCP
4. O Gateway bate no OIDC da Cognito e **valida as chaves do JWT**
5. O Gateway **chama a REST API alvo** usando sua Role segura do IAM

A API de destino fica protegida e só aceita o IAM do Gateway, enquanto o Gateway sabe de forma absoluta **quem** está logado. Extraímos também o `sub` do JWT para ser o `actor_id` da Memória da AWS.

Isso nos prepara para o **Módulo 6**, onde o Cedar bloqueará ações com base neste JWT.

> **Documentation:** [AgentCore Identity](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)

---

## Coletando variáveis da Infraestrutura

Precisamos coletar os IDs gerados pela stack do CloudFormation (Roles do IAM, ID da REST API e o provedor do Cognito).

In [ ]:
import sys; sys.path.insert(0, '..')
import boto3
from shared import utils

region = utils.get_region()
cfn = utils.get_all_cfn_outputs()

# Values from the CloudFormation prerequisites stack
gateway_role_arn = cfn.get("GatewayRoleArn") or cfn.get("GatewayServiceRoleArn")
rest_api_id = cfn.get("ApiGatewayRestApiId") or cfn.get("TaskApiRestApiId")
user_pool_id = cfn.get("UserPoolId") or cfn.get("CognitoUserPoolId")
cognito_client_id = cfn.get("UserPoolClientId") or cfn.get("CognitoClientId")

oidc_discovery_url = (
    f"https://cognito-idp.{region}.amazonaws.com/{user_pool_id}"
    f"/.well-known/openid-configuration"
) if user_pool_id else None

print(f"Region:            {region}")
print(f"Gateway Role ARN:  {gateway_role_arn}")
print(f"REST API ID:       {rest_api_id}")
print(f"User Pool ID:      {user_pool_id}")
print(f"Cognito Client ID: {cognito_client_id}")
print(f"OIDC Discovery:    {oidc_discovery_url}")

---

## Instanciando o Gateway

Usaremos a API `create_gateway` para subir esse componente. Configuramos:
- **`protocolType: MCP`** -- Comunicação no padrão da indústria
- **`authorizerType: CUSTOM_JWT`** -- Bloqueio e exigência do token JWT
- **`discoveryUrl`** -- O caminho da nuvem OIDC para validação das chaves RSA
- **`allowedAudience`** -- Valida a audiência rigorosa (ID do App Client)

In [ ]:
from botocore.exceptions import ClientError

control = boto3.client("bedrock-agentcore-control", region_name=region)

try:
    resp = control.create_gateway(
        name="aria-gateway",
        description="AgentCore Gateway for Aria — routes MCP tool requests to backend APIs",
        roleArn=gateway_role_arn,
        protocolType="MCP",
        authorizerType="CUSTOM_JWT",
        authorizerConfiguration={
            "customJWTAuthorizer": {
                "discoveryUrl": oidc_discovery_url,
                "allowedAudience": [cognito_client_id],
            }
        },
    )
    gateway_id = resp["gatewayId"]
    print(f"Gateway created: {gateway_id}")
    print(f"   URL: {resp.get('gatewayUrl', '')}")
    print(f"   Status: {resp['status']}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Gateway already exists -- looking it up...")
        paginator = control.get_paginator("list_gateways")
        for page in paginator.paginate():
            for gw in page.get("items", []):
                if gw["name"] == "aria-gateway":
                    gateway_id = gw["gatewayId"]
                    detail = control.get_gateway(gatewayIdentifier=gateway_id)
                    print(f"Found: {gateway_id}")
                    print(f"   URL: {detail.get('gatewayUrl', '')}")
                    break
    else:
        raise

### Aguardando o Gateway operar

O deploy de um Gateway é assíncrono. O polling aguardará o estado de `READY`.

In [ ]:
import time

for i in range(30):
    gw = control.get_gateway(gatewayIdentifier=gateway_id)
    status = gw["status"]
    print(f"  [{i*5}s] Status: {status}")
    if status == "READY":
        gateway_url = gw["gatewayUrl"]
        gateway_arn = gw.get("gatewayArn", "")
        print(f"\nGateway ready!")
        print(f"   URL: {gateway_url}")
        print(f"   ARN: {gateway_arn}")
        break
    if status in ("CREATE_FAILED", "FAILED"):
        print(f"Failed: {gw.get('statusReasons', '')}")
        break
    time.sleep(5)

---

## Embutindo o Target da API

Um **target** do Gateway aponta o MCP para um serviço que realmente exista. Ligaremos na REST API que foi provisionada na stack original.

Ponto chave da configuração:
- **`apiGateway`** — Faz a descoberta automática via `restApiId`.
- **`toolOverrides`** — Renomeia de maneira semântica as ferramentas (tools) mapeadas para facilitar a interpretação pela IA.
- **`toolFilters`** — Filtra as rotas liberadas por segurança estrita.
- **`GATEWAY_IAM_ROLE`** — Credencial IAM remota outboud que o Gateway assumirá.

In [ ]:
try:
    target_resp = control.create_gateway_target(
        gatewayIdentifier=gateway_id,
        name="TaskApi",
        description="Task Management REST API — CRUD operations for user tasks",
        targetConfiguration={
            "mcp": {
                "apiGateway": {
                    "restApiId": rest_api_id,
                    "stage": "prod",
                    "apiGatewayToolConfiguration": {
                        "toolOverrides": [
                            {"path": "/tasks", "method": "GET", "name": "list_tasks",
                             "description": "List all tasks for the current user"},
                            {"path": "/tasks", "method": "POST", "name": "create_task",
                             "description": "Create a new task. Requires 'title' in JSON body."},
                            {"path": "/tasks/{id}", "method": "PUT", "name": "update_task",
                             "description": "Update an existing task by ID."},
                            {"path": "/tasks/{id}", "method": "DELETE", "name": "delete_task",
                             "description": "Delete a task by ID."},
                        ],
                        "toolFilters": [
                            {"filterPath": "/tasks", "methods": ["GET", "POST"]},
                            {"filterPath": "/tasks/{id}", "methods": ["PUT", "DELETE"]},
                        ],
                    },
                }
            }
        },
        credentialProviderConfigurations=[
            {"credentialProviderType": "GATEWAY_IAM_ROLE"}
        ],
    )
    target_id = target_resp["targetId"]
    print(f"Target created: {target_id}")

except ClientError as e:
    if e.response["Error"]["Code"] in ("ConflictException", "ValidationException"):
        print("Target already exists")
    else:
        raise

### Salvando a Configuração do Gateway

In [ ]:
utils.save_config("gateway", {
    "gateway_id": gateway_id,
    "gateway_url": gateway_url,
    "gateway_arn": gateway_arn,
    "region": region,
})
print("Gateway config saved for later modules")

---
## Enable Tracing for Gateway

Enable **Tracing** on the Gateway resource so that MCP tool requests and policy evaluations appear in the AgentCore Observability dashboard.

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Gateways**
2. Select the **aria-gateway** resource
3. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

4. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

---

## Deploy com Gateway Acoplado

Vamos disparar um novo pacote do agente contendo as variáveis ambientais da `MEMORY_ID` e o novo `GATEWAY_ENDPOINT`.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.deploy_agent import deploy
from shared.utils import load_config

memory_cfg = load_config("memory")

env_vars = {}
if memory_cfg:
    env_vars["MEMORY_ID"] = memory_cfg["memory_id"]
env_vars["GATEWAY_ENDPOINT"] = gateway_url

result = deploy(
    agent_dir="agent",
    env_vars=env_vars,
)
runtime_arn = result["runtime_arn"]

---

## Testando a Gestão de Tarefas (Target API)

O Gateway utiliza o `CUSTOM_JWT`. Logo, precisamos injetar o token JWT na rotina de teste `test_agent`.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import test_agent

jwt_token = test_agent.get_test_token()

# Create a task
result = test_agent.invoke(
    "Create a task: Learn about AgentCore Gateway",
    jwt_token=jwt_token,
)

### Listar tarefas

In [ ]:
# List tasks — same session so Aria has conversation context
result = test_agent.invoke(
    "List all my tasks",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)

### Atualizar Tarefa para concluída

In [ ]:
# Complete a task
result = test_agent.invoke(
    "Mark the AgentCore Gateway task as completed",
    session_id=result["session_id"],
    jwt_token=jwt_token,
)


### Conversando Interativamente na CLI

Você pode abrir o CLI para conversar. Como agora temos Auth forte na AWS, passe a flag `--auth`:

```bash
cd /workshop/05-gateway-identity
python ../shared/chat.py --auth
```

Tente criar ou ler as tarefas de forma orgânica e conversacional.

> **Documentation:** [Authenticate and authorize with Inbound Auth and Outbound Auth](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/identity.html)

---

## O que teremos no próximo lab?

O Gateway está operante! A falha arquitetural do momento é: **qualquer pessoa logada pode fazer qualquer coisa** -- criar, deletar e bagunçar todas as APIs do endpoint.

No **Módulo 6: Política**, aplicaremos as **Políticas Cedar** no Gateway como um escudo impenetrável de regra de negócio.

---

## Progress

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("05")

---

**Next up: [Module 6 -- Enforce Policies with Cedar](../06-policy/notebook.ipynb)**